# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and preprocess the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) accessible at the provided URL and includes multiple record sets and fields for comprehensive exploration with entity references by `@id`.

### Dataset Source
Croissant Schema: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define URL of the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
ds = mlc.Dataset(croissant_url)

# Get and display basic metadata
meta = ds.metadata
print(f"{meta.name}: {meta.description}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), their fields, and the corresponding `@id` for referencing in subsequent operations.

In [ ]:
# List available record sets and their fields by `@id`
from collections import defaultdict

record_set_ids = []
record_set_fields = defaultdict(list)

for rs in ds.record_sets:
    record_set_ids.append(rs.id)
    print(f"RecordSet @id: {rs.id}")
    for field in rs.fields:
        print(f"  Field: @id: {field.id} | Name: {field.name} | DataType: {getattr(field, 'dataType', 'Unknown')}")
        record_set_fields[rs.id].append(field.id)
print(f"\nTotal RecordSets found: {len(record_set_ids)}")

## 3. Data Extraction
Load data from each record set into `pandas.DataFrame` for analysis. All entity references use `@id` as listed above.

In [ ]:
# Extract all records from each discovered record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(ds.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {rs_id}")
        print("Columns:", df.columns.tolist())
        print(df.head(2))
        print()
if not dataframes:
    print("No dataframes created, please check available record sets.")

## 4. Exploratory Data Analysis (EDA)
Select a numeric field by its `@id` from one of the record sets and perform data processing operations: filtering, normalization, and (optionally) grouping. All field and record set references are by `@id`.

In [ ]:
# Example EDA on a selected record set and numeric field
# Please adjust the record_set_id and numeric_field_id as per the previous overview outputs
# Here we demonstrate how to select (replace IDs as needed for real data)

# Pick the first record set as an example
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]
    print(f"Example RecordSet @id: {example_record_set_id}")
    print("Available columns:", df.columns.tolist())

    # Try to find a numeric field automatically if known, else use placeholder
    numeric_field_id = None
    for col in df.columns:
        if 'likelihood' in col.lower() or 'coef' in col.lower() or 'value' in col.lower() or 'numeric' in col.lower():
            numeric_field_id = col
            break
    if not numeric_field_id:
        # Fallback: pick the first float/integer column, else the first column
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if not numeric_field_id:
        numeric_field_id = df.columns[0]
    print(f"Using numeric field '@id': {numeric_field_id}")

    # Filtering: filter for values above mean as an example
    try:
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
    except Exception:
        # If not numeric, skip filtering
        threshold = None
        filtered_df = df.copy()
    print(f"Filtered records (where {numeric_field_id} > {threshold}):")
    print(filtered_df.head())

    # Normalization
    try:
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized '{numeric_field_id}':")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    except Exception as e:
        print(f"Could not normalize {numeric_field_id}: {e}")

    # Grouping by first non-numeric column if available
    group_field = None
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped statistics by {group_field}:")
        print(grouped_df.head())
    else:
        print("No categorical field found for grouping.")
else:
    print("No records found to perform EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and its relationship to a grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if dataframes:
    df = dataframes[example_record_set_id]
    field = numeric_field_id
    plt.figure(figsize=(7,4))
    df[field].hist(bins=20)
    plt.title(f'Distribution of {field}')
    plt.xlabel(field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        # boxplot by group field
        plt.figure(figsize=(8, 4))
        df.boxplot(column=field, by=group_field)
        plt.title(f'{field} by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(field)
        plt.show()
else:
    print("No dataframe available for visualization.")

## 6. Conclusion
We demonstrated how to access and explore the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id`. We loaded metadata, reviewed record sets and fields, extracted records to DataFrame, performed EDA (filtering, normalizing, grouping), and produced basic visualizations.

- To perform further analysis or modeling, reference entity `@id`s as shown in the overview and extraction steps.
- For best reproducibility, re-run the notebook after updating the `record_set_id` and `numeric_field_id` with those printed from your dataset overview.

**Note:** All references to dataset record sets, fields, and columns in this notebook use the canonical `@id` values, ensuring consistency and clarity according to the Croissant schema specification.